# AI_PARSE_DOCUMENT — PDF Invoice Data Extraction

## Use Case Overview

`AI_PARSE_DOCUMENT` extracts structured information from unstructured documents — PDFs, images, scanned forms — using a vision-capable LLM. No OCR configuration, no parsing libraries, no layout rules required.

**SA use case:** Automate invoice processing, contract review, form digitisation, and compliance document extraction — all within Snowflake, using standard SQL.

**Dataset:** 3 synthetic PDF invoices generated with `generate_invoices.py` and uploaded to an internal Snowflake stage.

**Prerequisites:** Run `generate_invoices.py` before this notebook:
```bash
pip install fpdf2
python generate_invoices.py
```

**What we'll demonstrate:**
1. Parsing a PDF invoice with `AI_PARSE_DOCUMENT`
2. Extracting specific fields (vendor, amount, dates, line items)
3. Loading all parsed invoices into a structured table
4. Querying the structured results

In [ ]:
import os
import pandas as pd
import snowflake.connector

conn = snowflake.connector.connect(
    connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME") or "genai-labs"
)
conn.cursor().execute("USE DATABASE GENAI_LEARNING")
conn.cursor().execute("USE SCHEMA PUBLIC")

staged_files = pd.read_sql("LIST @GENAI_LEARNING.PUBLIC.invoice_pdfs", conn)
print(f"PDFs in stage: {len(staged_files)}")
staged_files

## Step 2 — Parse a Single PDF

`AI_PARSE_DOCUMENT(TO_FILE('@stage', 'filename.pdf'), 'application/pdf')` returns a VARIANT containing the full document text and structure extracted by the LLM.

Note: `TO_FILE` creates a file reference pointing to a staged file. The document never leaves Snowflake.

In [ ]:
raw_parse = pd.read_sql("""
    SELECT
        AI_PARSE_DOCUMENT(
            TO_FILE('@GENAI_LEARNING.PUBLIC.invoice_pdfs', 'INV-2024-001.pdf'),
            'application/pdf'
        ) AS parsed_doc
""", conn)
print("Raw parsed output:")
print(raw_parse["PARSED_DOC"][0])

## Step 3 — Extract Specific Fields with AI_EXTRACT

Combine `AI_PARSE_DOCUMENT` with `AI_EXTRACT` to pull specific business fields from the parsed text.

In [ ]:
extracted = pd.read_sql("""
    WITH parsed AS (
        SELECT
            'INV-2024-001.pdf' AS filename,
            AI_PARSE_DOCUMENT(
                TO_FILE('@GENAI_LEARNING.PUBLIC.invoice_pdfs', 'INV-2024-001.pdf'),
                'application/pdf'
            ):content::STRING AS doc_text
    )
    SELECT
        filename,
        AI_EXTRACT(doc_text, [
            'vendor_name',
            'invoice_number',
            'invoice_date',
            'due_date',
            'total_amount_due',
            'bill_to_company'
        ]) AS fields
    FROM parsed
""", conn)
print("Extracted fields:")
print(extracted["FIELDS"][0])

## Step 4 — Process All Invoices and Load to Table

Process all staged PDFs in a single SQL statement and load results into the `invoice_parse_results` table.

In [ ]:
conn.cursor().execute("""
    INSERT INTO invoice_parse_results (invoice_file, raw_extraction, vendor_name, invoice_number, invoice_date, due_date, total_amount)
    WITH filenames AS (
        SELECT f.value::STRING AS fname
        FROM TABLE(RESULT_SCAN(LAST_QUERY_ID())) t, LATERAL FLATTEN(INPUT => ARRAY_CONSTRUCT(
            'INV-2024-001.pdf', 'INV-2024-002.pdf', 'INV-2024-003.pdf'
        )) f
    ),
    parsed AS (
        SELECT
            fname,
            AI_PARSE_DOCUMENT(
                TO_FILE('@GENAI_LEARNING.PUBLIC.invoice_pdfs', fname),
                'application/pdf'
            ):content::STRING AS doc_text
        FROM filenames
    ),
    extracted AS (
        SELECT
            fname,
            AI_EXTRACT(doc_text, ['vendor_name','invoice_number','invoice_date','due_date','total_amount_due']) AS fields
        FROM parsed
    )
    SELECT
        fname,
        fields,
        fields:vendor_name::STRING,
        fields:invoice_number::STRING,
        fields:invoice_date::STRING,
        fields:due_date::STRING,
        fields:total_amount_due::STRING
    FROM extracted
""")

results_df = pd.read_sql("SELECT invoice_file, vendor_name, invoice_number, invoice_date, due_date, total_amount FROM invoice_parse_results", conn)
results_df

## Step 5 — Interpretation & SA Tips

**SA tips:**
- `AI_PARSE_DOCUMENT` handles PDFs, images (PNG, JPEG, TIFF), and scanned documents
- Combine with a `DIRECTORY TABLE` on the stage to auto-process new files as they arrive
- Use `STREAMS` + `TASKS` to build an automated invoice ingestion pipeline: new PDF → parse → load to structured table
- For high-volume processing, batch files using a stored procedure to avoid hitting concurrency limits
- The `:content` field of the parsed VARIANT contains the full extracted text; use `AI_EXTRACT` on it for specific fields